# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ahmedsamymohamad/flyrank_internship_starter/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

The output of the probability model is divided into three actionable tiers. Reason codes are generated based on historical volume to inform the editor whether the page is a 'High-Value Defender' or 'Low-Volume Shifter'.

In [1]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# We quickly re-run the prediction to generate the queue
df = pd.read_parquet('../outputs/capstone_features.parquet')
feature_cols = ['clicks_feat', 'impressions_feat', 'avg_pos_feat', 'search_volume', 'competition', 'word_count']
for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

test_df = df[df['split'] == 'test'].copy()
train_df = df[df['split'] == 'train']

clf = RandomForestClassifier(n_estimators=50, max_depth=6, random_state=42)
clf.fit(train_df[feature_cols], train_df['is_declining_label'])

test_df['risk_score'] = clf.predict_proba(test_df[feature_cols])[:, 1]

def assign_action(score):
    if score > 0.8: return "Immediate Review / Rewrite"
    if score > 0.5: return "Monitor in GSC"
    return "Maintain (Do nothing)"

def assign_reason(row):
    if row['risk_score'] > 0.8 and row['clicks_feat'] > 1000:
        return "High Volume Decay Risk"
    elif row['risk_score'] > 0.8:
        return "Low Volume Stagnation"
    return "Stable"

test_df['action'] = test_df['risk_score'].apply(assign_action)
test_df['reason_code'] = test_df.apply(assign_reason, axis=1)

queue = test_df[['content_hash_id', 'clicks_feat', 'risk_score', 'action', 'reason_code']].sort_values('risk_score', ascending=False)
display(queue.head())

,content_hash_id,clicks_feat,risk_score,action,reason_code
475879,content_d37c679d80751802,15.0,0.636258,Monitor in GSC,Stable
474291,content_168b91c64165000a,18.0,0.635807,Monitor in GSC,Stable
346865,content_d703f6d6febd5142,13.0,0.635754,Monitor in GSC,Stable
505042,content_cfb77f68ed35a1ba,11.0,0.634287,Monitor in GSC,Stable
346676,content_b31eb956a0c002b3,10.0,0.633585,Monitor in GSC,Stable


## 2. Intended use and limits

**Intended Use:** Content editors should load the top 20 pages of the queue every Monday morning to decide which pages to manually audit or update. 

**Limits:** The model does not understand content quality or search intent. It purely identifies historical momentum patterns. It is a decision-support tool, not an absolute truth.

In [2]:
# Text only section
print("Intended use and limits defined.")

Intended use and limits defined.


## 3. Human review + the no-go list

**Human Review:** Editors must verify if the decline is due to a known seasonal event (e.g., 'Christmas Gifts' page dropping in January) before rewriting.

**No-Go List:** 
- DO NOT automatically delete or redirect pages based on a high risk score.
- DO NOT feed these pages to an LLM for automatic rewriting without human approval.

In [3]:
# Text only section
print("Review rules defined.")

Review rules defined.


## 4. Monitoring / retrain triggers

**Monitoring:** We will track the precision of the high-risk queue. If less than 20% of the 'High Risk' pages actually drop in traffic the following month, the model is over-predicting.

**Retrain Triggers:**
- A major Google Core Algorithm Update.
- When the overall base-rate of site decline shifts significantly (e.g., site-wide penalty).

In [4]:
# Text only section
print("Triggers defined.")

Triggers defined.


## 5. Exports for the paper

We export the ranked queue to `work/outputs/action_queue.csv` for the capstone paper. We don't commit this CSV to the repo to respect data privacy rules.

In [5]:
import os
os.makedirs('../outputs', exist_ok=True)
queue.to_csv('../outputs/action_queue.csv', index=False)
print("Exported queue to work/outputs/action_queue.csv")

Exported queue to work/outputs/action_queue.csv


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.